# GRPO 深度解析：数学原理

> **TIP**: 本节深入 GRPO 的技术细节和数学推导，作者为 Shirin Yamani。如果你对数学推导感到陌生，可以先聚焦于概念理解，再逐步深入数学部分。

GRPO 的核心思想是：**通过在同一组生成结果中进行比较来优化策略模型，而不是训练一个独立的价值模型（Critic）**。这种方式大幅降低了计算成本。

GRPO 可以应用于任何**可验证任务**（即可以判断答案是否正确的任务），例如数学推理问题（可直接对比标准答案）。

## GRPO 算法三步骤

### Step 1：分组采样（Group Sampling）

**目标**：为每个问题生成多个候选答案，构成一个多样化的比较组

对于每个问题 $q$，从当前策略 $\pi_{\theta_{old}}$ 中生成 $G$ 个输出：

$$\{o_1, o_2, o_3, ..., o_G\} \sim \pi_{\theta_{old}}$$

其中 $G$ 通常设为 8。

**示例**：

问题 $q$：计算 $2 + 2 \times 6$

生成 8 个输出（G=8）：
```
{o_1: 14(正确), o_2: 16(错误), o_3: 10(错误), ..., o_8: 14(正确)}
```

注意某些答案正确（14），某些错误（16 或 10）。这种多样性对下一步至关重要。

**Comment**

$\pi_{\theta_{old}}(o_i|q)$ 表示旧策略 $\pi_{\theta_{old}}$ 下，生成 $q$ 时，输出 $o_i$ 的概率。

### Step 2：优势计算（Advantage Calculation）

**目标**：确定哪些回答优于组内平均水平

**奖励分配**：给每个输出分配奖励分数 $r_i$（可以是规则函数，也可以是奖励模型）
- 正确答案：$r_i = 1$
- 错误答案：$r_i = 0$

**优势值计算公式**：

$$A_i = \frac{r_i - \text{mean}(\{r_1, r_2, ..., r_G\})}{\text{std}(\{r_1, r_2, ..., r_G\})}$$

**示例**（8 个输出中 3 个正确）：

奖励序列：$r = [\underbrace{1, 1, 1}_{3\ 个正确}, \underbrace{0, 0, 0, 0, 0}_{5\ 个错误}]$

**均值**：

$$\text{mean}(r_i) = \frac{3 \times 1 + 5 \times 0}{8} = \frac{3}{8} = 0.375$$

**标准差**（样本标准差，$n-1$ 为分母）：

$$\text{var} = \frac{3 \times (1 - 0.375)^2 + 5 \times (0 - 0.375)^2}{8 - 1} = \frac{1.171875 + 0.703125}{7} = \frac{1.875}{7} \approx 0.2679$$

$$\text{std} = \sqrt{0.2679} \approx 0.5175$$

**优势值**：

| 统计量 | 数值 |
|--------|------|
| 组内均值 | $\text{mean}(r_i) = 0.375$ |
| 标准差 | $\text{std}(r_i) \approx 0.5175$ |
| 正确答案的优势值 | $A_i = (1 - 0.375) / 0.5175 \approx +1.208$ |
| 错误答案的优势值 | $A_i = (0 - 0.375) / 0.5175 \approx -0.725$ |

**理解**：
- $A_i > 0$：该回答优于组内平均水平 → 应该被**强化**
- $A_i < 0$：该回答低于组内平均水平 → 应该被**抑制**

> **NOTE**：正确答案越稀少（均值越低），每个正确答案的优势值绝对值越大，模型被激励生成正确答案的信号越强。

### Step 3：策略优化（Policy Optimization）

**目标**：更新模型，使其倾向于生成优势值高的回答

GRPO 的目标函数：

$$J_{GRPO}(\theta) = \mathbb{E}\left[\min\left(\text{ratio} \cdot A_i,\ \text{clip}(\text{ratio}, 1-\varepsilon, 1+\varepsilon) \cdot A_i\right)\right] - \beta \cdot D_{KL}(\pi_\theta || \pi_{ref})$$

其中 $\text{ratio} = \pi_\theta(o_i|q) / \pi_{\theta_{old}}(o_i|q)$

#### 目标函数三大组件详解

##### 组件一：概率比率（Probability Ratio）

$$\text{ratio} = \frac{\pi_\theta(o_i|q)}{\pi_{\theta_{old}}(o_i|q)}$$

这个比率衡量新策略相对于旧策略的变化程度：
- $\text{ratio} > 1$：新模型对输出 $o_i$ 的概率比旧模型**更高**（该回答被强化）
- $\text{ratio} < 1$：新模型对输出 $o_i$ 的概率比旧模型**更低**（该回答被抑制）

##### 组件二：裁剪函数（Clip Function）

$$\text{clip}\left(\frac{\pi_\theta(o_i|q)}{\pi_{\theta_{old}}(o_i|q)},\ 1-\varepsilon,\ 1+\varepsilon\right)$$

裁剪函数将概率比率限制在 $[1-\varepsilon, 1+\varepsilon]$ 范围内，防止策略更新过于激进。

**示例（ε = 0.2，比值范围限制在 [0.8, 1.2]）**：

- **情形 1**：新策略对某回答概率从 0.5 → 0.9
  - 原始比率：$0.9 / 0.5 = 1.8$
  - 裁剪后：$\min(1.8, 1.2) = 1.2$（防止更新过大）

- **情形 2**：新策略对某回答概率从 0.5 → 0.2
  - 原始比率：$0.2 / 0.5 = 0.4$
  - 裁剪后：$\max(0.4, 0.8) = 0.8$（防止下降过多）

**以下三个作用都不是 clip 函数单独产生的，而是通过 $\min(\text{ratio} \cdot A_i,\ \text{clip}(\text{ratio}) \cdot A_i)$ 整体协同涌现的**：

**作用一：鼓励新模型强化旧模型低估但质量高的回答**

前提场景：旧模型对某好答案概率低（$\pi_{\theta_{old}}$ 小），$A_i > 0$

- 不加 clip 时：ratio 可能迅速跳到很大，梯度信号 $\text{ratio} \cdot A_i$ 随之暴增 → 单步更新过大，训练震荡，被迫用极小学习率
- 加了 clip 后：ratio 被限制在 $[1,\ 1+\varepsilon]$，梯度信号上界变成 $(1+\varepsilon) \cdot A_i$ → 每步幅度受控，可用正常学习率在多步训练中稳定地渐进强化

「鼓励」的真正含义：clip 通过消除梯度爆炸，让训练可以持续多步往正确方向走。不是 clip 在「推」，而是 clip 扶住了「推」的过程不翻车。

**作用二：限制对高概率但质量差的回答的维持**

前提场景：旧模型对某坏答案概率高（$\pi_{\theta_{old}}$ 大），$A_i < 0$

子场景 1：新模型错误地继续提升该坏答案（ratio > 1+ε）：

$$\min(\underbrace{1.5 \times (-0.725)}_{-1.088},\ \underbrace{1.2 \times (-0.725)}_{-0.870}) = -1.088 \quad \leftarrow \text{未裁剪项胜出，完整惩罚保留}$$

→ clip 项被 min 丢弃，完整惩罚不被削弱，限制了对坏答案的持续强化。此处起作用的是 **min 的选择逻辑**，clip 本身反而是「被丢弃的那一方」。

子场景 2：新模型正确抑制坏答案，但抑制过猛（ratio < 1-ε）：

$$\min(\underbrace{0.3 \times (-0.725)}_{-0.218},\ \underbrace{0.8 \times (-0.725)}_{-0.580}) = -0.580 \quad \leftarrow \text{裁剪项胜出，防止一步压崩}$$

→ clip 将过于激进的抑制限制到 0.8 倍，防止参数更新步子过大导致不稳定。

**作用三：确保每步更新幅度在可控范围内**

这是 clip **唯一直接产生**的效果：

$$\text{clip}(\text{ratio},\ 1-\varepsilon,\ 1+\varepsilon) \Rightarrow \text{ratio} \in [0.8,\ 1.2]$$

无论模型此前有多大偏差，单步梯度更新最多只能把策略改变 $\varepsilon$ 倍。这保证了 on-policy 方法的核心稳定性：训练数据的采样分布与当前策略分布始终不会差距过大。

| 作用 | clip 的直接贡献 | 真正的主要机制 |
|------|---------------|--------------|
| 鼓励强化好答案 | 防止 ratio 过大导致梯度爆炸 | 稳定多步训练，间接效果 |
| 限制维持坏答案 | 防止 ratio 过小时抑制过猛 | min + $A_i < 0$ 的协同，直接效果 |
| 控制更新幅度 | 直接将 ratio 限定在 $[1-\varepsilon,\ 1+\varepsilon]$ | **clip 唯一直接产生的效果** |

##### 组件三：KL 散度（KL Divergence）

$$\beta \cdot D_{KL}(\pi_\theta || \pi_{ref})$$

KL 散度惩罚项防止新策略过度偏离参考策略（通常是更新前的模型）。

**数学定义**：
$$D_{KL}(P || Q) = \sum_{x \in X} P(x) \cdot \log\frac{P(x)}{Q(x)}$$

**β 参数的作用**：

| β 值 | 效果 | 风险 |
|------|------|------|
| **较大** | 强 KL 约束，策略变化慢 | 适应慢，可能无法充分探索 |
| **较小** | 弱 KL 约束，策略变化快 | 可能产生奖励黑客行为，输出不稳定 |
| **推荐值** | DeepSeekMath 论文设 β = 0.04 | 平衡性能与稳定性 |

> **WARNING**: KL 散度过小时，模型可能会「钻空子」：找到奖励函数的漏洞，生成能获得高奖励但实际无意义的输出（称为 Reward Hacking）。

#### min(裁剪前, 裁剪后) 深度拆解：代入 3/8 正确示例

##### 回顾 Step 2 的数值

从 3/8 正确的示例中，我们已经得到：

| 答案类型 | 奖励 $r_i$ | 优势值 $A_i$ |
|---------|-----------|-------------|
| **正确答案**（3个） | 1 | **+1.208** |
| **错误答案**（5个） | 0 | **-0.725** |

取 $\varepsilon = 0.2$，即裁剪范围为 $[0.8,\ 1.2]$。

---

##### 五种情形逐一拆解

**情形 A：正确答案，模型「过度强化」（ratio = 1.5）**

新模型把正确答案的概率大幅提升了。

$$\text{未裁剪项} = 1.5 \times 1.208 = 1.812$$
$$\text{裁剪项} = \text{clip}(1.5,\ 0.8,\ 1.2) \times 1.208 = 1.2 \times 1.208 = 1.450$$
$$\min(1.812,\ 1.450) = \boxed{1.450} \quad \leftarrow \text{裁剪项胜出}$$

结论：虽然答案正确，但更新幅度被压制——模型已经够好了，不需要再「用力拉」。

---

**情形 B：正确答案，模型适度强化（ratio = 1.1）**

$$\text{未裁剪项} = 1.1 \times 1.208 = 1.329$$
$$\text{裁剪项} = \text{clip}(1.1,\ 0.8,\ 1.2) \times 1.208 = 1.1 \times 1.208 = 1.329$$
$$\min(1.329,\ 1.329) = \boxed{1.329} \quad \leftarrow \text{两项相等，正常更新}$$

结论：在安全范围 $[0.8,\ 1.2]$ 内，裁剪不起作用，正常鼓励正确答案。

---

**情形 C：正确答案，模型反向移动（ratio = 0.5，方向错误）**

新模型把正确答案的概率降低了。

$$\text{未裁剪项} = 0.5 \times 1.208 = 0.604$$
$$\text{裁剪项} = \text{clip}(0.5,\ 0.8,\ 1.2) \times 1.208 = 0.8 \times 1.208 = 0.966$$
$$\min(0.604,\ 0.966) = \boxed{0.604} \quad \leftarrow \text{未裁剪项胜出}$$

结论：正确答案概率在下降，目标函数值很低 → 梯度会把模型「拉回来」，纠正错误方向。

> **为什么「未裁剪项胜出」能产生纠错梯度？**
>
> 训练目标是**最大化** $J_{GRPO}$，优化器沿梯度方向更新 $\theta$。当前胜出的是未裁剪项：
>
> $$J \ni \text{ratio} \cdot A_i = \frac{\pi_\theta(o_i|q)}{\pi_{\theta_{old}}(o_i|q)} \cdot A_i$$
>
> 对 $\theta$ 求梯度：
>
> $$\frac{\partial J}{\partial \theta} \propto A_i \cdot \frac{1}{\pi_{\theta_{old}}} \cdot \frac{\partial \pi_\theta(o_i|q)}{\partial \theta}$$
>
> 由于 $A_i = +1.208 > 0$，梯度方向要求 $\pi_\theta(o_i|q)$ **增大**，即 ratio 往 1 方向回升，即把概率拉回正确方向。
>
> **若裁剪项胜出（假设情形）**：ratio = 0.5 < 0.8，clip 将其钉死在常数 0.8，则：
>
> $$\frac{\partial J}{\partial \theta} \propto \frac{\partial (0.8 \cdot A_i)}{\partial \theta} = 0$$
>
> 梯度归零，模型收不到任何纠错信号。**未裁剪项胜出正是为了保留 ratio 对 $\theta$ 的依赖关系，使梯度非零，纠错才得以发生。**

---

**情形 D：错误答案，模型正确抑制（ratio = 0.5）**

新模型把错误答案的概率降低了（正确方向）。

$$\text{未裁剪项} = 0.5 \times (-0.725) = -0.363$$
$$\text{裁剪项} = \text{clip}(0.5,\ 0.8,\ 1.2) \times (-0.725) = 0.8 \times (-0.725) = -0.580$$
$$\min(-0.363,\ -0.580) = \boxed{-0.580} \quad \leftarrow \text{裁剪项胜出}$$

结论：错误答案已经被压制了，裁剪限制「惩罚力度」不能太大——别矫枉过正。

---

**情形 E：错误答案，模型反向强化（ratio = 1.5，危险！）**

新模型把错误答案的概率反而提高了。

$$\text{未裁剪项} = 1.5 \times (-0.725) = -1.088$$
$$\text{裁剪项} = \text{clip}(1.5,\ 0.8,\ 1.2) \times (-0.725) = 1.2 \times (-0.725) = -0.870$$
$$\min(-1.088,\ -0.870) = \boxed{-1.088} \quad \leftarrow \text{未裁剪项胜出}$$

结论：错误答案概率在上升，目标函数值很低（负数更大）→ 梯度强力惩罚，把模型「拉回来」。

---

##### min 操作的设计哲学

| 情形 | ratio 方向 | $A_i$ 符号 | 谁胜出 | 作用 |
|------|-----------|-----------|--------|------|
| 正确答案，过度强化 | ratio > 1.2 | + | 裁剪项 | 防止更新过猛 |
| 正确答案，适度强化 | ratio ∈ [0.8, 1.2] | + | 相等 | 正常鼓励 |
| 正确答案，反向下降 | ratio < 0.8 | + | 未裁剪项 | 施压纠正 |
| 错误答案，正确抑制 | ratio < 0.8 | − | 裁剪项 | 防止矫枉过正 |
| 错误答案，反向上升 | ratio > 1.2 | − | 未裁剪项 | 强力惩罚 |

**一句话总结**：`min` 操作的设计哲学是「**好事不能做过头，坏事必须被纠正**」。

---

##### 完整数值流程（3/8 正确，ratio = 1.1，ε = 0.2，β = 0.04）

$$E[\cdots] = \frac{3 \times \overbrace{1.329}^{\text{正确答案}} + 5 \times \overbrace{(-0.653)}^{\text{错误答案}}}{8} = \frac{3.987 - 3.265}{8} \approx +0.090$$

$$J_{GRPO} = 0.090 - 0.04 \times D_{KL} > 0 \quad \Rightarrow \quad \text{梯度方向：提升正确答案概率，降低错误答案概率} \checkmark$$

> **NOTE**：注意到错误答案的 ratio 这里取 0.9（$0.9 \times (-0.725) = -0.653$），即新模型已经在轻微抑制错误答案。整体期望为正，说明这一步更新方向正确。

## 完整 Worked Example

以问题「计算 $2 + 2 \times 6$」为例，逐步演示完整的 GRPO 流程。

In [3]:
import torch
import torch.nn.functional as F

# =============================================================
# Step 1：分组采样（Group Sampling）
# 假设模型对问题「2x+1 when x=2, 答案是?」生成了 8 个候选答案
# =============================================================

print("Step 1: 分组采样")
print("-" * 40)
# 一个问题的 8 个候选答案，正确答案是 5
# 格式：一组，8 个生成结果
responses = [5, 6, 7, 5, 3, 5, 8, 5]   # 问题：2x+1 when x=2，正确答案是 5

print(f"问题的候选答案：{responses}（正确答案：5）")

Step 1: 分组采样
----------------------------------------
问题的候选答案：[5, 6, 7, 5, 3, 5, 8, 5]（正确答案：5）


In [4]:
# =============================================================
# Step 2：奖励计算和优势值计算（Advantage Calculation）
# =============================================================

print("Step 2: 奖励计算与优势值计算")
print("-" * 40)

# 基于正确性分配奖励：正确=1，错误=0
# responses = [5, 6, 7, 5, 3, 5, 8, 5]
#  rewards  = [1, 0, 0, 1, 0, 1, 0, 1]  # 5正确，6错误，7错误，5正确，3错误，5正确，8错误，5正确

# 将奖励整理为一个 tensor，形状 (B*G,) = (8,)
# B=1（问题数量），G=8（每题生成数量）
rewards = torch.tensor([1, 0, 0, 1, 0, 1, 0, 1], dtype=torch.float32)
num_generations = 8  # 每个问题生成 8 个候选答案

# 按组重塑：形状从 (B*G,) = (8,) 变为 (B, G) = (1, 8)
rewards_grouped = rewards.view(-1, num_generations)
print(f"分组后的奖励矩阵 (B×G):\n{rewards_grouped}")

# 计算每组的均值和标准差，形状均为 (B,) = (1,)
mean_grouped_rewards = rewards_grouped.mean(dim=1)
std_grouped_rewards = rewards_grouped.std(dim=1)
print(f"\n每组均值：{mean_grouped_rewards}")
print(f"每组标准差：{std_grouped_rewards}")

# 广播均值和标准差以匹配展平后的 rewards 形状
# 从 (B,) = (1,) 扩展为 (B*G,) = (8,)
# repeat_interleave：将每个元素重复 G=8 次
mean_grouped_rewards = mean_grouped_rewards.repeat_interleave(num_generations, dim=0)
std_grouped_rewards = std_grouped_rewards.repeat_interleave(num_generations, dim=0)
print(f"\n广播后的均值：{mean_grouped_rewards}")
print(f"广播后的标准差：{std_grouped_rewards}")

# 计算优势值：(奖励 - 组内均值) / 组内标准差
# +1e-8 防止除以零
advantages = (rewards - mean_grouped_rewards) / (std_grouped_rewards + 1e-8)
print(f"\n各候选答案的优势值：{advantages}")
print("  >0 表示优于组内平均 → 应被强化")
print("  <0 表示低于组内平均 → 应被抑制")

Step 2: 奖励计算与优势值计算
----------------------------------------
分组后的奖励矩阵 (B×G):
tensor([[1., 0., 0., 1., 0., 1., 0., 1.]])

每组均值：tensor([0.5000])
每组标准差：tensor([0.5345])

广播后的均值：tensor([0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000])
广播后的标准差：tensor([0.5345, 0.5345, 0.5345, 0.5345, 0.5345, 0.5345, 0.5345, 0.5345])

各候选答案的优势值：tensor([ 0.9354, -0.9354, -0.9354,  0.9354, -0.9354,  0.9354, -0.9354,  0.9354])
  >0 表示优于组内平均 → 应被强化
  <0 表示低于组内平均 → 应被抑制


In [4]:
# =============================================================
# Step 3：策略更新（Policy Optimization）
# 展示 GRPO 目标函数的计算方式
# =============================================================

print("Step 3: 策略更新（GRPO 目标函数）")
print("-" * 40)

# 将优势值调整形状为 (B*G, 1) = (8, 1)，以便与 logps 的形状匹配
# B=1（问题数量），G=8（每题生成数量）
advantages_2d = advantages.unsqueeze(1)
print(f"优势值形状调整后 (B*G, 1):\n{advantages_2d}")

# 模拟 per_token_logps 和 new_per_token_logps
# 在实际训练中，这些来自模型对生成 token 的 log 概率
# 这里使用随机值模拟，形状 (B*G, seq_len) = (8, 1)  # B=1, G=8
torch.manual_seed(42)  # 固定随机种子，保证结果可复现
per_token_logps = torch.randn(8, 1) * 0.5      # 旧策略的 log 概率
new_per_token_logps = per_token_logps + torch.randn(8, 1) * 0.2  # 新策略（略有变化）

# 计算概率比率（在 log 空间中相减等价于除法）
# ratio = π_θ(o_i|q) / π_θ_old(o_i|q)
# = exp(log π_θ(o_i|q) - log π_θ_old(o_i|q))
ratio = torch.exp(new_per_token_logps - per_token_logps)
print(f"\n概率比率 (ratio) 示例（前4个）:\n{ratio[:4]}")

# 裁剪系数 ε
epsilon = 0.2

# 计算两个损失项（取较大值确保保守更新）
# pg_losses1: 未裁剪的策略梯度损失（负号是因为我们要最大化目标）
pg_losses1 = -advantages_2d * ratio

# pg_losses2: 裁剪后的策略梯度损失
pg_losses2 = -advantages_2d * torch.clamp(ratio, 1.0 - epsilon, 1.0 + epsilon)

# 取两个损失中的较大值（即目标函数中的 min 操作）
# 取较大的损失 = 取较保守的策略更新
pg_loss = torch.max(pg_losses1, pg_losses2)

print(f"\n策略梯度损失（裁剪前）示例（前4个）:\n{pg_losses1[:4]}")
print(f"\n策略梯度损失（裁剪后）示例（前4个）:\n{pg_losses2[:4]}")
print(f"\n最终策略梯度损失（取最大值）示例（前4个）:\n{pg_loss[:4]}")

print()
print("=" * 50)
print("GRPO 完整流程总结：")
print("1. 为每个 prompt 生成 G 个候选答案（Group Sampling）")
print("2. 用奖励函数评分，归一化得到优势值（Advantage）")
print("3. 用裁剪目标函数 + KL 惩罚更新策略（Policy Update）")

Step 3: 策略更新（GRPO 目标函数）
----------------------------------------
优势值形状调整后 (B*G, 1):
tensor([[ 0.9354],
        [-0.9354],
        [-0.9354],
        [ 0.9354],
        [-0.9354],
        [ 0.9354],
        [-0.9354],
        [ 0.9354]])

概率比率 (ratio) 示例（前4个）:
tensor([[1.0967],
        [1.0549],
        [1.1129],
        [1.1757]])

策略梯度损失（裁剪前）示例（前4个）:
tensor([[-1.0259],
        [ 0.9868],
        [ 1.0410],
        [-1.0998]])

策略梯度损失（裁剪后）示例（前4个）:
tensor([[-1.0259],
        [ 0.9868],
        [ 1.0410],
        [-1.0998]])

最终策略梯度损失（取最大值）示例（前4个）:
tensor([[-1.0259],
        [ 0.9868],
        [ 1.0410],
        [-1.0998]])

GRPO 完整流程总结：
1. 为每个 prompt 生成 G 个候选答案（Group Sampling）
2. 用奖励函数评分，归一化得到优势值（Advantage）
3. 用裁剪目标函数 + KL 惩罚更新策略（Policy Update）


## 用 transformers 加载模型并实际生成（完整示例）

下面展示如何加载真实模型并完成分组采样过程：

In [5]:
# 完整示例：加载模型并进行分组采样
# 注意：首次运行会自动下载模型权重（约 3GB），需要网络连接
# 实际训练中由 GRPOTrainer 自动处理完整流程

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

# --------------------------------------------------------
# 加载模型和 tokenizer
# --------------------------------------------------------
model_name = "Qwen/Qwen2-Math-1.5B"  # 使用 Qwen 数学模型演示
print(f"正在加载模型：{model_name} ...")
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model.eval()
print("模型加载完成。")

# 自动检测并使用 GPU（如有）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用设备：{device}")
model.to(device)

# --------------------------------------------------------
# 准备输入 prompt
# --------------------------------------------------------
prompt = "Solve y = 2x + 1 for x = 2, y = "  # 正确答案：5

inputs = tokenizer(prompt, return_tensors="pt", padding=True)
input_ids = inputs["input_ids"].to(device)           # 形状：(1, prompt_len)
attention_mask = inputs["attention_mask"].to(device)
prompt_len = input_ids.shape[1]
print(f"\nPrompt：\"{prompt}\"")
print(f"Prompt token 数：{prompt_len}")

# --------------------------------------------------------
# 分组采样：生成 G = 8 个候选答案
# --------------------------------------------------------
num_generations = 8   # 每个 prompt 生成 8 个候选答案（GRPO 典型设置）

print(f"\n开始分组采样（G={num_generations}）...")
with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,                              # 形状：(1, prompt_len)
        attention_mask=attention_mask,
        max_new_tokens=5,                                 # 每次最多生成 5 个 token
        num_return_sequences=num_generations,             # 总共生成 8 个序列
        do_sample=True,                                   # 采样（非贪心，增加多样性）
        top_k=50,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

# outputs 形状：(num_generations, prompt_len + max_new_tokens)
# 只解码新生成的 token（去掉 prompt 部分）
generated_ids = outputs[:, prompt_len:]   # 形状：(8, max_new_tokens)
generated_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

print(f"\n模型实际生成的 {num_generations} 个候选答案：")
print(f"{'序号':<6} {'生成文本':<20} {'是否正确'}")
print("-" * 40)
for i, text in enumerate(generated_texts):
    answer = text.strip()
    # 简单判断：是否包含 "5"
    correct = "✓" if "5" in answer else "✗"
    print(f"  o_{i+1:<3} {repr(answer):<20} {correct}")

print()
print(f"注：以上是模型在 temperature=0.9 下的真实采样结果，每次运行结果会不同。")
print(f"    GRPO 正是利用这 {num_generations} 个结果的奖励差异来计算优势值，进而更新策略。")

正在加载模型：Qwen/Qwen2-Math-1.5B ...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

模型加载完成。
当前使用设备：cpu

Prompt："Solve y = 2x + 1 for x = 2, y = "
Prompt token 数：19

开始分组采样（G=8）...

模型实际生成的 8 个候选答案：
序号     生成文本                 是否正确
----------------------------------------
  o_1   '0, and y ='         ✗
  o_2   '1, and z ='         ✗
  o_3   '3, and z ='         ✗
  o_4   '2, and x ='         ✗
  o_5   '5, and y ='         ✓
  o_6   '4, and y ='         ✗
  o_7   '4, and z ='         ✗
  o_8   '0, and x ='         ✗

注：以上是模型在 temperature=0.9 下的真实采样结果，每次运行结果会不同。
    GRPO 正是利用这 8 个结果的奖励差异来计算优势值，进而更新策略。


In [6]:
# --------------------------------------------------------
# Step 2（真实数据）：奖励计算与优势值计算
# 基于上面模型实际生成的 8 个候选答案
# --------------------------------------------------------

print("Step 2（真实数据）: 奖励计算与优势值计算")
print("-" * 40)

# 奖励函数：去掉空格后，首字符为 "5" 则视为正确
# 比 "5" in text 更严格，避免把 "65" 误判为正确
rewards_real = []
for text in generated_texts:
    answer = text.strip()
    is_correct = answer.startswith("5")
    rewards_real.append(1.0 if is_correct else 0.0)

rewards_tensor = torch.tensor(rewards_real, dtype=torch.float32)

print(f"各候选答案奖励（首字符为 '5' 得 1 分，否则 0 分）：")
for i, (text, r) in enumerate(zip(generated_texts, rewards_real)):
    symbol = "✓" if r == 1.0 else "✗"
    print(f"  o_{i+1}: {repr(text.strip()[:15]):<18} 奖励={r:.0f}  {symbol}")

print(f"\n正确答案数量：{int(rewards_tensor.sum())} / {len(rewards_real)}")

# 计算组内均值和标准差
mean_reward = rewards_tensor.mean()
std_reward = rewards_tensor.std()
print(f"组内均值：{mean_reward:.4f}")
print(f"组内标准差：{std_reward:.4f}")

# 边界情况：所有答案相同时标准差为 0，优势值全为 0（无梯度信号）
if std_reward < 1e-8:
    print("\n警告：标准差为 0（所有答案奖励相同），优势值全为 0，本轮无有效梯度信号")
    advantages_real = torch.zeros_like(rewards_tensor)
else:
    advantages_real = (rewards_tensor - mean_reward) / (std_reward + 1e-8)

print(f"\n各候选答案优势值：")
for i, (text, r, a) in enumerate(zip(generated_texts, rewards_real, advantages_real)):
    symbol = "✓" if r == 1.0 else "✗"
    print(f"  o_{i+1}: {repr(text.strip()[:15]):<18} 优势值={a:+.4f}  {symbol}")

print()
print("优势值 > 0：该答案优于组内平均 → 训练时应被强化")
print("优势值 < 0：该答案低于组内平均 → 训练时应被抑制")

Step 2（真实数据）: 奖励计算与优势值计算
----------------------------------------
各候选答案奖励（首字符为 '5' 得 1 分，否则 0 分）：
  o_1: '0, and y ='       奖励=0  ✗
  o_2: '1, and z ='       奖励=0  ✗
  o_3: '3, and z ='       奖励=0  ✗
  o_4: '2, and x ='       奖励=0  ✗
  o_5: '5, and y ='       奖励=1  ✓
  o_6: '4, and y ='       奖励=0  ✗
  o_7: '4, and z ='       奖励=0  ✗
  o_8: '0, and x ='       奖励=0  ✗

正确答案数量：1 / 8
组内均值：0.1250
组内标准差：0.3536

各候选答案优势值：
  o_1: '0, and y ='       优势值=-0.3536  ✗
  o_2: '1, and z ='       优势值=-0.3536  ✗
  o_3: '3, and z ='       优势值=-0.3536  ✗
  o_4: '2, and x ='       优势值=-0.3536  ✗
  o_5: '5, and y ='       优势值=+2.4749  ✓
  o_6: '4, and y ='       优势值=-0.3536  ✗
  o_7: '4, and z ='       优势值=-0.3536  ✗
  o_8: '0, and x ='       优势值=-0.3536  ✗

优势值 > 0：该答案优于组内平均 → 训练时应被强化
优势值 < 0：该答案低于组内平均 → 训练时应被抑制


In [7]:
# --------------------------------------------------------
# Step 3（真实数据）：计算 log 概率与 GRPO 目标函数
# --------------------------------------------------------

print("Step 3（真实数据）: 策略优化（GRPO 目标函数）")
print("=" * 60)

# ══════════════════════════════════════════════════════════════
# 3a. 旧策略 log 概率（π_θ_old）
# 用模型前向传播，计算每个生成 token 在旧策略下的 log 概率
# 这些 log 概率在采样后立即计算，之后冻结作为基准
# ══════════════════════════════════════════════════════════════
print("\n[3a] 计算旧策略 log 概率 (π_θ_old)")
print("-" * 40)

# outputs 形状：(8, prompt_len + max_new_tokens)
with torch.no_grad():
    logits = model(input_ids=outputs).logits   # (8, full_len, vocab_size)

# 关键：logits[:, i, :] 预测的是 token[:, i+1]
# 因此 prompt 最后一个位置的 logit（index = prompt_len-1）预测第 1 个生成 token
# 生成部分的 logits 索引范围：[prompt_len-1 : -1]（共 max_new_tokens 个）
gen_logits = logits[:, prompt_len - 1 : -1, :]  # (8, max_new_tokens, vocab_size)
gen_tokens = outputs[:, prompt_len:]             # (8, max_new_tokens)

log_probs_all = F.log_softmax(gen_logits, dim=-1)   # (8, max_new_tokens, vocab_size)
# gather：从 vocab_size 维度取出每个实际生成 token 的 log 概率
per_token_logps = log_probs_all.gather(
    2, gen_tokens.unsqueeze(-1)
).squeeze(-1).cpu().float()   # (8, max_new_tokens)；.float() 将 BFloat16 转为 float32

# 打印每个序列的逐 token log 概率，并附上生成文本
print(f"{'序号':<6} {'生成文本':<20} {'每个 token 的 log 概率（旧策略）':<40} {'序列均值'}")
print("-" * 90)
for i in range(len(generated_texts)):
    token_ids = gen_tokens[i].tolist()
    token_strs = [repr(tokenizer.decode([t])) for t in token_ids]
    logps = per_token_logps[i].tolist()
    # 格式：token/logp 对
    detail = "  ".join(f"{ts}:{lp:.2f}" for ts, lp in zip(token_strs, logps))
    mean_lp = per_token_logps[i].mean().item()
    symbol = "✓" if rewards_real[i] == 1.0 else "✗"
    print(f"  o_{i+1} {symbol}  {repr(generated_texts[i].strip()[:12]):<20} {detail:<50} 均值={mean_lp:.3f}")

# ══════════════════════════════════════════════════════════════
# 3b. 新策略 log 概率（π_θ）
# 实际训练中，这是 optimizer.step() 后的模型重新前向传播的结果
# 这里用随机扰动模拟，以展示完整计算流程
# ══════════════════════════════════════════════════════════════
print("\n[3b] 模拟新策略 log 概率 (π_θ，梯度更新后)")
print("-" * 40)

torch.manual_seed(0)
new_per_token_logps = per_token_logps + torch.randn_like(per_token_logps) * 0.15

print(f"{'序号':<6} {'每个 token 的 log 概率变化（旧→新）':<60} {'序列均值变化'}")
print("-" * 90)
for i in range(len(generated_texts)):
    old_lps = per_token_logps[i].tolist()
    new_lps = new_per_token_logps[i].tolist()
    delta = [n - o for n, o in zip(new_lps, old_lps)]
    detail = "  ".join(f"{d:+.2f}" for d in delta)
    symbol = "✓" if rewards_real[i] == 1.0 else "✗"
    mean_delta = sum(delta) / len(delta)
    print(f"  o_{i+1} {symbol}  Δlog_p: {detail:<55} 均值Δ={mean_delta:+.3f}")

# ══════════════════════════════════════════════════════════════
# 3c. 概率比率 ratio
# ratio = π_θ / π_θ_old = exp(log π_θ - log π_θ_old)
# ratio > 1：新策略提升了该 token 的概率
# ratio < 1：新策略降低了该 token 的概率
# ratio = 1：策略未发生变化
# ══════════════════════════════════════════════════════════════
print("\n[3c] 概率比率 ratio = π_θ / π_θ_old")
print("-" * 40)

ratio = torch.exp(new_per_token_logps - per_token_logps)  # (8, max_new_tokens)

print(f"{'序号':<6} {'优势值 A_i':<12} {'各 token ratio':<50} {'序列均值 ratio'}")
print("-" * 90)
for i in range(len(generated_texts)):
    ratios = ratio[i].tolist()
    detail = "  ".join(f"{r:.3f}" for r in ratios)
    mean_r = ratio[i].mean().item()
    adv_val = advantages_real[i].item()
    symbol = "✓" if rewards_real[i] == 1.0 else "✗"
    print(f"  o_{i+1} {symbol}  A_i={adv_val:+.4f}   {detail:<55} 均值={mean_r:.3f}")

print(f"\n  裁剪范围：[1-ε, 1+ε] = [0.8, 1.2]（ε={0.2}）")
print(f"  超出范围的 ratio 会被裁剪，防止单步更新幅度过大")

# ══════════════════════════════════════════════════════════════
# 3d. 裁剪 PPO 目标函数
# pg_loss = -A_i × min(ratio, clip(ratio, 1-ε, 1+ε))
# 取负号是因为优化器做梯度下降（minimize loss），
# 而 GRPO 目标是最大化期望奖励（maximize J），两者方向相反
# ══════════════════════════════════════════════════════════════
print("\n[3d] 裁剪 PPO 目标：min(未裁剪项, 裁剪项)")
print("-" * 40)

epsilon = 0.2

adv = advantages_real.unsqueeze(1)  # (8, 1) → 广播到 (8, max_new_tokens)

pg_loss_unclipped = -adv * ratio
pg_loss_clipped   = -adv * torch.clamp(ratio, 1 - epsilon, 1 + epsilon)
pg_loss_per_token = torch.max(pg_loss_unclipped, pg_loss_clipped)  # 取保守项

pg_loss_per_seq = pg_loss_per_token.mean(dim=1)  # 在 token 维度上取均值，形状 (8,)

print(f"{'序号':<6} {'优势值 A_i':<12} {'未裁剪均值':<14} {'裁剪后均值':<14} {'最终损失':<12} {'说明'}")
print("-" * 90)
for i in range(len(generated_texts)):
    adv_val = advantages_real[i].item()
    unc = pg_loss_unclipped[i].mean().item()
    clp = pg_loss_clipped[i].mean().item()
    fin = pg_loss_per_seq[i].item()
    symbol = "✓" if rewards_real[i] == 1.0 else "✗"
    # 说明：损失为负意味着梯度方向是「提升该序列概率」（对应目标函数上升）
    note = "强化（损失↓）" if fin < 0 else "抑制（损失↑）"
    print(f"  o_{i+1} {symbol}  {adv_val:+.4f}     {unc:+.4f}        {clp:+.4f}        {fin:+.4f}    {note}")

# ══════════════════════════════════════════════════════════════
# 3e. KL 散度惩罚
# 防止新策略过度偏离旧策略（避免奖励黑客行为）
# 使用无偏估计：D_KL ≈ exp(log_old - log_new) - (log_old - log_new) - 1
# ══════════════════════════════════════════════════════════════
print("\n[3e] KL 散度惩罚（防止策略偏移过大）")
print("-" * 40)

kl_per_token = (
    torch.exp(per_token_logps - new_per_token_logps)
    - (per_token_logps - new_per_token_logps)
    - 1
)
kl_per_seq = kl_per_token.mean(dim=1)  # 每个序列的 KL 均值

print(f"{'序号':<6} {'每个序列的 KL 散度':<20} {'说明'}")
print("-" * 50)
for i in range(len(generated_texts)):
    kl_val = kl_per_seq[i].item()
    symbol = "✓" if rewards_real[i] == 1.0 else "✗"
    note = "策略几乎未变" if kl_val < 0.01 else ("小幅偏移" if kl_val < 0.05 else "偏移较大")
    print(f"  o_{i+1} {symbol}  KL={kl_val:.5f}              {note}")

kl_loss = kl_per_token.mean()
print(f"\n  全局平均 KL 散度：{kl_loss:.5f}")

# ══════════════════════════════════════════════════════════════
# 3f. 汇总：GRPO 总损失
# J_GRPO = E[min(ratio·A_i, clip(ratio)·A_i)] - β·D_KL
# 转为损失（取负号）：loss = -E[...] + β·D_KL
# ══════════════════════════════════════════════════════════════
beta = 0.04  # KL 惩罚系数（DeepSeekMath 推荐值）
total_loss = pg_loss_per_token.mean() + beta * kl_loss

print(f"\n{'='*60}")
print(f"GRPO 目标函数汇总")
print(f"{'='*60}")
print(f"  策略梯度损失（期望）：    {pg_loss_per_token.mean():+.4f}")
print(f"  KL 散度（全局均值）：     {kl_loss:.5f}")
print(f"  KL 惩罚项（β={beta}×KL）：{beta * kl_loss:+.5f}")
print(f"  ─────────────────────────────────")
print(f"  总 GRPO 损失：            {total_loss:+.4f}")
print()
print("  → 实际训练时执行：loss.backward() + optimizer.step()")
print("  → GRPOTrainer 自动循环：采样 → 奖励 → 优势 → 更新策略")

Step 3（真实数据）: 策略优化（GRPO 目标函数）

[3a] 计算旧策略 log 概率 (π_θ_old)
----------------------------------------
序号     生成文本                 每个 token 的 log 概率（旧策略）                   序列均值
------------------------------------------------------------------------------------------
  o_1 ✗  '0, and y ='         '0':-2.03  ',':-0.21  ' and':-0.20  ' y':-0.66  ' =':-0.02 均值=-0.625
  o_2 ✗  '1, and z ='         '1':-1.78  ',':-0.57  ' and':-0.25  ' z':-1.34  ' =':-0.01 均值=-0.791
  o_3 ✗  '3, and z ='         '3':-1.16  ',':-0.32  ' and':-0.17  ' z':-0.68  ' =':-0.02 均值=-0.467
  o_4 ✗  '2, and x ='         '2':-2.03  ',':-0.82  ' and':-0.25  ' x':-0.37  ' =':-0.03 均值=-0.702
  o_5 ✓  '5, and y ='         '5':-2.66  ',':-0.33  ' and':-0.10  ' y':-1.92  ' =':-0.02 均值=-1.006
  o_6 ✗  '4, and y ='         '4':-2.03  ',':-0.25  ' and':-0.13  ' y':-1.78  ' =':-0.01 均值=-0.841
  o_7 ✗  '4, and z ='         '4':-2.03  ',':-0.25  ' and':-0.13  ' z':-1.41  ' =':-0.02 均值=-0.768
  o_8 ✗  '0, and x ='         '0':-2.03  '

## 本节小结

恭喜！你已经掌握了 GRPO 的数学原理。回顾要点：

1. **分组采样**：GRPO 通过对一组生成结果进行组内比较来确定哪些更好，无需单独的价值模型

2. **优势计算**：通过标准化奖励（减均值除标准差）来识别哪些回答优于或劣于平均水平

3. **策略更新**：使用带裁剪的目标函数 + KL 散度惩罚，确保学习过程稳定可控

这种方法对数学推理任务特别有效，因为正确性可以被客观验证。

### 关键参数总结

| 参数 | 说明 | 典型值 |
|------|------|--------|
| $G$（组大小） | 每个 prompt 的候选数 | 4~16 |
| $\varepsilon$（裁剪范围） | 控制策略更新幅度 | 0.2 |
| $\beta$（KL 惩罚系数） | 控制偏离参考策略的程度 | 0.04 |

### 参考资料

1. [RLHF Book by Nathan Lambert](https://github.com/natolambert/rlhf-book)
2. [DeepSeek-V3 Technical Report](https://huggingface.co/papers/2412.19437)
3. [DeepSeekMath Paper](https://huggingface.co/papers/2402.03300)
4. [TRL GRPO Trainer 源码](https://github.com/huggingface/trl/blob/main/trl/trainer/grpo_trainer.py)

---

**下一节**：我们将学习如何使用 TRL 库的 `GRPOTrainer` 实际实现 GRPO 训练，无需手动实现数学细节。